# Hướng dẫn Train YOLOv5n-cls (Classification) - JetRacer Smart City (Summer 2026)

Notebook này hướng dẫn quy trình tiền xử lý phân tách dữ liệu (split train/val) và huấn luyện mô hình phân loại **YOLOv5n-cls** trên Google Colab. Mô hình phân loại chạy cực nhanh trên Jetson Nano (chỉ ~2-5ms), đáp ứng tốt giới hạn thời gian xử lý <= 300ms của đề bài.

## Bước 1: Chuẩn bị môi trường GPU & Cài đặt dependencies

In [ ]:
# Kiểm tra GPU hoạt động trên Colab
!nvidia-smi

# Clone YOLOv5 repository từ Ultralytics
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5

# Cài đặt các thư viện cần thiết
!pip install -r requirements.txt

## Bước 2: Tải lên và giải nén dữ liệu (.zip)

Chạy cell dưới đây. Nếu bạn đã kéo thả file `.zip` vào thư mục của Colab, nó sẽ tự động giải nén. Nếu chưa, Colab sẽ hiển thị nút bấm để bạn chọn file tải lên.

In [ ]:
import shutil
import os
import zipfile
from google.colab import files

# Quét các file .zip có sẵn trong thư mục hiện tại (thư mục cha /content/)
parent_dir = os.path.abspath('..')
zip_files = [f for f in os.listdir(parent_dir) if f.endswith('.zip')]

if not zip_files:
    print("Chưa phát hiện file .zip nào được kéo thả lên Colab. Hãy chọn tải lên file dữ liệu:")
    uploaded = files.upload()
    # Di chuyển file vừa upload ra thư mục cha để đồng bộ đường dẫn
    for filename in uploaded.keys():
        shutil.move(filename, os.path.join(parent_dir, filename))
    zip_files = [f for f in os.listdir(parent_dir) if f.endswith('.zip')]

if zip_files:
    zip_path = os.path.join(parent_dir, zip_files[0])
    print(f"Đang tiến hành giải nén file: {zip_path}...")
    
    # Giải nén vào thư mục /content/data/
    os.makedirs(os.path.join(parent_dir, 'data'), exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(os.path.join(parent_dir, 'data'))
        
    print("Giải nén hoàn tất!")
    print("Các thư mục trong /content/data/:", os.listdir(os.path.join(parent_dir, 'data')))
else:
    print("Lỗi: Không tìm thấy file dữ liệu nào để giải nén.")

## Bước 3: Phân chia cấu trúc lại tập dữ liệu (Split Dataset)

YOLOv5 Classification yêu cầu dữ liệu được chia sẵn thành thư mục `train` và `val` theo định dạng:
```
dataset_split/
├── train/
│   ├── class_A/
│   └── class_B/
└── val/
    ├── class_A/
    └── class_B/
```

Đoạn mã sau giúp tự động phân chia tập dữ liệu của bạn (tỷ lệ 80% train, 20% val):

In [ ]:
import shutil
import random

def split_classification_dataset(src_dir, dest_dir, split_ratio=0.8):
    """
    Chia dữ liệu từ thư mục raw thành train/val
    """
    classes = [d for d in os.listdir(src_dir) if os.path.isdir(os.path.join(src_dir, d))]
    
    train_dir = os.path.join(dest_dir, 'train')
    val_dir = os.path.join(dest_dir, 'val')
    
    for cls in classes:
        os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
        os.makedirs(os.path.join(val_dir, cls), exist_ok=True)
        
        cls_src = os.path.join(src_dir, cls)
        files = [f for f in os.listdir(cls_src) if os.path.isfile(os.path.join(cls_src, f))]
        
        random.seed(42)
        random.shuffle(files)
        
        split_idx = int(len(files) * split_ratio)
        train_files = files[:split_idx]
        val_files = files[split_idx:]
        
        for f in train_files:
            shutil.copy(os.path.join(cls_src, f), os.path.join(train_dir, cls, f))
        for f in val_files:
            shutil.copy(os.path.join(cls_src, f), os.path.join(val_dir, cls, f))
            
    print(f"Phân chia hoàn tất! Dữ liệu lưu tại: {dest_dir}")

# Tự động phân chia tập Biển Báo từ dữ liệu đã giải nén ở trên
# Thư mục nguồn sau giải nén có thể là '/content/data' hoặc chứa thư mục lồng. Hãy cấu hình đường dẫn nguồn phù hợp:
src_path = '../data'
# Kiểm tra nếu folder được lồng bên trong thư mục giải nén
if 'data' in os.listdir(src_path) and os.path.isdir(os.path.join(src_path, 'data')):
    src_path = os.path.join(src_path, 'data')

# Nếu thư mục chứa folder 'Biển báo'
if 'Biển báo' in os.listdir(src_path):
    split_classification_dataset(os.path.join(src_path, 'Biển báo'), '../data/Biển_báo_split', split_ratio=0.8)
else:
    # Trường hợp zip nén trực tiếp các nhãn ngoài cùng
    split_classification_dataset(src_path, '../data/Biển_báo_split', split_ratio=0.8)

## Bước 4: Huấn luyện mô hình YOLOv5n-cls

Chúng ta dùng script `classify/train.py` của YOLOv5 dành riêng cho bài toán phân loại.

> **CHÚ Ý**: Để thay đổi số lượng Epochs (vòng lặp huấn luyện), hãy điều chỉnh giá trị sau thuộc tính `--epochs` (mặc định là `50` ở bên dưới).

In [ ]:
# === CẤU HÌNH THÔNG SỐ TRAIN TẠI ĐÂY ===
# Thay đổi --epochs 50 thành số lượng bạn mong muốn (ví dụ: --epochs 100)
!python classify/train.py \
  --model yolov5n-cls.pt \
  --data ../data/Biển_báo_split \
  --epochs 50 \
  --imgsz 224 \
  --device 0

## Bước 4.5: Vẽ biểu đồ hàm Loss để đánh giá mô hình

Chạy cell dưới đây để vẽ biểu đồ Loss Curves (Train Loss và Validation Loss) nhằm xác định xem mô hình có bị quá khớp (overfitting), thiếu khớp (underfitting) hay đã hội tụ hoàn toàn để tối ưu hóa số epoch.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Tìm thư mục chạy mới nhất (exp, exp2, ...)
runs_dir = 'runs/train-cls'
exps = [d for d in os.listdir(runs_dir) if d.startswith('exp')] if os.path.exists(runs_dir) else []
latest_exp = sorted(exps, key=lambda x: int(x.replace('exp', '')) if x.replace('exp', '') else 1)[-1] if exps else 'exp'
results_csv = os.path.join(runs_dir, latest_exp, 'results.csv')

if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    # Xóa khoảng trắng thừa trong tên cột
    df.columns = [c.strip() for c in df.columns]
    
    plt.figure(figsize=(10, 6))
    
    # Tìm các cột chứa giá trị loss
    loss_cols = [col for col in df.columns if 'loss' in col.lower()]
    
    for col in loss_cols:
        plt.plot(df['epoch'], df[col], label=col, linewidth=2)
        
    plt.title(f'YOLOv5-cls Loss Curves ({latest_exp})', fontsize=14, fontweight='bold')
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(fontsize=11)
    plt.show()
else:
    print(f"Không tìm thấy file results.csv tại {results_csv}. Hãy chắc chắn lệnh train đã hoàn thành.")

## Bước 5: Xuất mô hình sang ONNX Opset 11

In [ ]:
# Tìm thư mục chạy mới nhất (exp, exp2, ...) để export đúng file best.pt vừa train
import os
runs_dir = 'runs/train-cls'
exps = [d for d in os.listdir(runs_dir) if d.startswith('exp')] if os.path.exists(runs_dir) else []
latest_exp = sorted(exps, key=lambda x: int(x.replace('exp', '')) if x.replace('exp', '') else 1)[-1] if exps else 'exp'
weights_path = os.path.join(runs_dir, latest_exp, 'weights/best.pt')

print(f"Exporting weights from: {weights_path}")
!python export.py --weights {weights_path} --include onnx --opset 11

## Bước 6: Tự động tải file `best.onnx` về máy tính

Chạy cell dưới đây để tự động tải file mô hình đã biên dịch định dạng ONNX về máy.

In [ ]:
from google.colab import files

# Xác định đường dẫn file ONNX vừa xuất ở exp mới nhất
onnx_file_path = os.path.join(runs_dir, latest_exp, 'weights/best.onnx')

if os.path.exists(onnx_file_path):
    print(f"Đang tải file: {onnx_file_path}")
    files.download(onnx_file_path)
else:
    # Dự phòng nếu chưa export ONNX, tải file PyTorch (.pt)
    pt_file_path = os.path.join(runs_dir, latest_exp, 'weights/best.pt')
    if os.path.exists(pt_file_path):
        print(f"Không tìm thấy ONNX. Đang tải file PyTorch (.pt): {pt_file_path}")
        files.download(pt_file_path)
    else:
        print("Lỗi: Không tìm thấy checkpoint nào trong thư mục weights!")

## Bước 6.5: Lưu file `best.onnx` lên Google Drive (Dự phòng)

Chạy cell dưới đây để mount Google Drive của bạn và tự động sao chép file mô hình `best.onnx` hoặc `best.pt` vào thư mục Drive để lưu trữ lâu dài.

In [ ]:
import os
import shutil
from google.colab import drive

try:
    # 1. Mount Google Drive
    print("Đang kết nối với Google Drive...")
    drive.mount('/content/drive')

    # 2. Định nghĩa thư mục lưu trữ trên Drive
    drive_save_dir = '/content/drive/MyDrive/yolov5_smartcity'
    os.makedirs(drive_save_dir, exist_ok=True)

    # 3. Đường dẫn file nguồn và đích
    onnx_file_path = os.path.join(runs_dir, latest_exp, 'weights/best.onnx')
    pt_file_path = os.path.join(runs_dir, latest_exp, 'weights/best.pt')

    if os.path.exists(onnx_file_path):
        dest_path = os.path.join(drive_save_dir, 'best.onnx')
        shutil.copy(onnx_file_path, dest_path)
        print(f"Thành công! Đã sao lưu: {onnx_file_path} -> {dest_path}")
    else:
        if os.path.exists(pt_file_path):
            dest_path = os.path.join(drive_save_dir, 'best.pt')
            shutil.copy(pt_file_path, dest_path)
            print(f"Không tìm thấy ONNX. Đã sao lưu file PyTorch (.pt): {pt_file_path} -> {dest_path}")
        else:
            print("Lỗi: Không tìm thấy file checkpoint nào trong thư mục weights!")
except Exception as e:
    print(f"Đã xảy ra lỗi khi kết nối hoặc lưu file lên Google Drive: {e}")

## Bước 7: Biên dịch sang TensorRT trên Jetson Nano

Khi đã có file `best.onnx` trên Jetson, chạy lệnh sau ở terminal của Jetson để compile sang `.engine` FP16:
```bash
trtexec --onnx=best.onnx --saveEngine=yolov5n_smartcity.engine --fp16 --workspace=1024
```